## 1. Imports and Setup

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

# Import SFT and GRPO
from simple_rl.algorithms import SFT, GRPO

# Import evaluation utilities
from simple_rl.evaluation.gsm8k import (
    load_gsm8k_dataset,
    prepare_gsm8k_prompts,
    prepare_gsm8k_for_sft,
    evaluate_on_gsm8k,
    demonstrate_model_responses
)

# Import reward functions
from simple_rl.rewards import (
    extract_answer_from_model_output,
    compute_math_rewards_batch,
)

# Import dataset caching
from simple_rl.utils.dataset_cache import (
    load_gsm8k_cache,
    save_gsm8k_cache,
    load_cot_cache,
    save_cot_cache,
)

# Import logging utility
from simple_rl.utils.notebook_logger import setup_notebook_logger, close_logger

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device detection
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    torch.mps.manual_seed(42)
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: mps


## 2. Configuration

In [2]:
# ============================================================
# Initialize Logging (console + file)
# ============================================================
logger, log_path = setup_notebook_logger("logs", "training")

logger.info("="*60)
logger.info("TRAINING RUN STARTED")
logger.info("="*60)
logger.info(f"📋 Log file: {log_path}")
logger.info("="*60)

# ============================================================
# Configuration
# ============================================================

# RUN_SFT 
RUN_SFT = False
if not RUN_SFT:
    sft_checkpoint_path = Path("checkpoints/pipeline_stages/01_after_sft") / "sft_complete.pt"

# Resume training configuration
CONTINUE_FROM = 0  #45 # Set to episode number to resume from GRPO checkpoint, 0 = start from SFT/base model

# Model configuration
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# System prompt for math formatting
SYSTEM_PROMPT = """
Respond in the following format:

<reasoning>
...
</reasoning>
<answer>
...
</answer>
"""

# Dataset cache directory (for deterministic loading)
DATASET_CACHE_DIR = "dataset_cache"

logger.info(f"Model: {MODEL_NAME}")
logger.info(f"System Prompt: {SYSTEM_PROMPT.strip()}")
logger.info(f"Dataset cache: {DATASET_CACHE_DIR}")
logger.info(f"Resume from episode: {CONTINUE_FROM if CONTINUE_FROM > 0 else 'None (starting fresh)'}")

TRAINING RUN STARTED
📋 Log file: logs/training_2025-10-14_230444.log
Model: Qwen/Qwen2.5-0.5B-Instruct
System Prompt: Respond in the following format:

<reasoning>
...
</reasoning>
<answer>
...
</answer>
Dataset cache: dataset_cache
Resume from episode: None (starting fresh)


## 3. Load and Prepare Dataset

In [3]:
# Load GSM8K dataset with caching for deterministic loading
# SFT gets first 500 examples, GRPO gets remaining examples (NO OVERLAP)

print("="*60)
print("Loading and Preparing Dataset (with caching)")
print("="*60)

# Define splits (used for cache key generation)
gsm8k_splits = {
    "sft_train": "train[:500]",
    "grpo_train": "train[500:]",
    "val": "test[200:1200]",
    "test": "test[:200]"
}

# Try to load from cache first
print("\n🔍 Checking for cached GSM8K dataset...")
cached_data = load_gsm8k_cache(
    cache_dir=DATASET_CACHE_DIR,
    splits=gsm8k_splits,
    system_prompt=SYSTEM_PROMPT
)

if cached_data is not None:
    # Load from cache (fast, deterministic)
    print("✓ Loading dataset from cache (deterministic)")
    (sft_train_data, sft_val_data, sft_test_data,
     grpo_train_prompts, grpo_train_answers,
     grpo_val_prompts, grpo_val_answers,
     grpo_test_prompts, grpo_test_answers) = cached_data
else:
    # Load from HuggingFace and process (slow, first time only)
    print("⏳ Cache not found, loading from HuggingFace and processing...")
    
    # Load SFT data (first 500 training examples)
    dataset_sft_train, dataset_val, dataset_test = load_gsm8k_dataset(
        train_split=gsm8k_splits["sft_train"],
        val_split=gsm8k_splits["val"],
        test_split=gsm8k_splits["test"]
    )
    
    # Load GRPO data (remaining training examples - NO OVERLAP)
    dataset_grpo_train, _, _ = load_gsm8k_dataset(
        train_split=gsm8k_splits["grpo_train"],
        val_split=gsm8k_splits["val"],
        test_split=gsm8k_splits["test"]
    )
    
    # Prepare data for SFT (prompts + completions)
    sft_train_data = prepare_gsm8k_for_sft(dataset_sft_train, SYSTEM_PROMPT)
    sft_val_data = prepare_gsm8k_for_sft(dataset_val, SYSTEM_PROMPT)
    sft_test_data = prepare_gsm8k_for_sft(dataset_test, SYSTEM_PROMPT)
    
    # Prepare data for GRPO (prompts + answers) - DIFFERENT training data
    grpo_train_prompts, grpo_train_answers = prepare_gsm8k_prompts(dataset_grpo_train, SYSTEM_PROMPT)
    grpo_val_prompts, grpo_val_answers = prepare_gsm8k_prompts(dataset_val, SYSTEM_PROMPT)
    grpo_test_prompts, grpo_test_answers = prepare_gsm8k_prompts(dataset_test, SYSTEM_PROMPT)
    
    # Save to cache for next time
    save_gsm8k_cache(
        cache_dir=DATASET_CACHE_DIR,
        splits=gsm8k_splits,
        system_prompt=SYSTEM_PROMPT,
        sft_train_data=sft_train_data,
        sft_val_data=sft_val_data,
        sft_test_data=sft_test_data,
        grpo_train_prompts=grpo_train_prompts,
        grpo_train_answers=grpo_train_answers,
        grpo_val_prompts=grpo_val_prompts,
        grpo_val_answers=grpo_val_answers,
        grpo_test_prompts=grpo_test_prompts,
        grpo_test_answers=grpo_test_answers
    )

print(f"\n{'='*60}")
print("Dataset prepared for full pipeline (NO OVERLAP)")
print(f"{'='*60}")
print(f"SFT Training samples: {len(sft_train_data['prompts'])} (train[:500])")
print(f"GRPO Training samples: {len(grpo_train_prompts)} (train[500:])")
print(f"Validation samples: {len(sft_val_data['prompts'])}")
print(f"Test samples: {len(sft_test_data['prompts'])}")

# Show example
print(f"\nSFT Example prompt:\n{sft_train_data['prompts'][0]}")
print(f"\nSFT Example completion:\n{sft_train_data['completions'][0][:200]}...")
print(f"\nGRPO Example prompt (different data):\n{grpo_train_prompts[0][:150]}...")
print(f"GRPO Example answer: {grpo_train_answers[0]}")

Loading and Preparing Dataset (with caching)

🔍 Checking for cached GSM8K dataset...

📂 Loading dataset from cache: dataset_cache/gsm8k_full_pipeline_fde9ec8cc6a6de8b.json
   Cache file size: 3.90 MB
   ✓ Cache loaded successfully
✓ Loading dataset from cache (deterministic)

Dataset prepared for full pipeline (NO OVERLAP)
SFT Training samples: 500 (train[:500])
GRPO Training samples: 6973 (train[500:])
Validation samples: 1000
Test samples: 200

SFT Example prompt:
Respond in the following format:

<reasoning>
...
</reasoning>
<answer>
...
</answer>
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

SFT Example completion:
<reasoning>
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
</reasoning>
<answer>
72
</answer>...

GRPO Example prompt (different data):
Respond in the following format:

<reasoning>
...
</re

In [4]:
# OPTIONAL: Download and use high-quality CoT dataset from reference notebook
# This dataset has 500 pre-generated reasoning chains that are cleaner than raw GSM8K
# Now with caching for fast, deterministic loading

import os
import hashlib
import tarfile
import requests
from datasets import load_dataset

def download_and_extract_cot_archive(url, extract_path="cot_archive"):
    """Download and extract the CoT archive if not already done."""
    archive_path = os.path.join(extract_path, "cot.tar.gz")
    if not os.path.exists(extract_path):
        os.makedirs(extract_path, exist_ok=True)
    
    if not os.path.exists(archive_path):
        print("Downloading CoT archive from reference notebook...")
        r = requests.get(url, stream=True)
        with open(archive_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
        print(f"✓ Downloaded to {archive_path}")
    
    # Extract the archive if not already extracted
    extract_dir = os.path.join(extract_path, "cot_files")
    if not os.path.exists(extract_dir):
        print("Extracting CoT archive...")
        with tarfile.open(archive_path, "r:gz") as tar:
            tar.extractall(path=extract_dir)
        print(f"✓ Extracted to {extract_dir}")
    
    return extract_dir

def prepare_cot_dataset_for_sft(num_examples=500):
    """
    Prepare high-quality CoT examples from the reference notebook's dataset.
    Returns data in the same format as prepare_gsm8k_for_sft().
    """
    cot_url = "https://github.com/aburkov/theLMbook/releases/download/v1.0.0/cot.tar.gz"
    extract_dir = download_and_extract_cot_archive(cot_url)
    
    # Load GSM8K to get questions
    data = load_dataset('gsm8k', 'main')["train"]
    
    prompts = []
    completions = []
    
    for example in data:
        question = example["question"].strip()
        
        # Compute the filename based on SHA-256 hash of the question
        filename = hashlib.sha256(question.encode()).hexdigest() + ".txt"
        file_path = os.path.join(extract_dir, filename)
        
        if os.path.exists(file_path):
            with open(file_path, "r", encoding="utf-8") as f:
                cot_output = f.read().strip()
            
            # Build prompt with system prompt
            prompt = f"{SYSTEM_PROMPT.strip()}\n{question}"
            prompts.append(prompt)
            completions.append(cot_output)
        
        if len(prompts) >= num_examples:
            break
    
    print(f"\n✓ Loaded {len(prompts)} high-quality CoT examples")
    print(f"  These have cleaner reasoning than raw GSM8K")
    
    return {"prompts": prompts, "completions": completions}

# Load high-quality CoT dataset with caching
print("\n🔍 Checking for cached CoT dataset...")
cot_num_examples = 500
sft_train_data_cot = load_cot_cache(
    cache_dir=DATASET_CACHE_DIR,
    system_prompt=SYSTEM_PROMPT,
    num_examples=cot_num_examples
)

if sft_train_data_cot is None:
    # Load from source and cache
    print("⏳ CoT cache not found, downloading and processing...")
    sft_train_data_cot = prepare_cot_dataset_for_sft(num_examples=cot_num_examples)
    
    # Save to cache
    save_cot_cache(
        cache_dir=DATASET_CACHE_DIR,
        system_prompt=SYSTEM_PROMPT,
        num_examples=cot_num_examples,
        sft_train_data_cot=sft_train_data_cot
    )
else:
    print(f"✓ Loaded {len(sft_train_data_cot['prompts'])} CoT examples from cache")

# Show sample
print(f"\n" + "="*60)
print("SAMPLE CoT EXAMPLE:")
print("="*60)
print(f"Prompt:\n{sft_train_data_cot['prompts'][0][:150]}...")
print(f"\nCompletion:\n{sft_train_data_cot['completions'][0][:300]}...")
print("="*60)

print(f"\n✓ CoT dataset ready!")
print(f"  To use it, replace 'sft_train_data' with 'sft_train_data_cot' in cell 12")
sft_train_data_cot["debug_boundary"] = True
print(f"  Example: sft_results = sft.train(train_data=sft_train_data_cot, ...)")


🔍 Checking for cached CoT dataset...

📂 Loading dataset from cache: dataset_cache/gsm8k_cot_5121dc302e064ba0.json
   Cache file size: 0.79 MB
   ✓ Cache loaded successfully
✓ Loaded 500 CoT examples from cache

SAMPLE CoT EXAMPLE:
Prompt:
Respond in the following format:

<reasoning>
...
</reasoning>
<answer>
...
</answer>
Natalia sold clips to 48 of her friends in April, and then she s...

Completion:
<reasoning>
In April, Natalia sold clips to 48 of her friends.

In May, she sold half as many clips as she did in April. 

To find out how many clips she sold in May, we need to divide the number of clips she sold in April by 2. 

48 clips / 2 = 24 clips

So, Natalia sold 24 clips in May.

To find o...

✓ CoT dataset ready!
  To use it, replace 'sft_train_data' with 'sft_train_data_cot' in cell 12
  Example: sft_results = sft.train(train_data=sft_train_data_cot, ...)


## 4. Initial Evaluation (Base Model - Before Training)

In [5]:
if RUN_SFT:
        # # Initialize SFT just for evaluation of base model
    print("Loading base model for initial evaluation...")
    sft_config_eval = {
        "model": {
            "model_name": MODEL_NAME,
            "max_length": 1024,
            "device": "auto",
        },
        "training": {"batch_size": 4, "learning_rate": 5e-5, "num_epochs": 1},
        "logging": {"log_interval": 50, "save_interval": 500},
        "validation": {"enabled": False},
        "wandb": {"enabled": False}
    }

    base_model_eval = SFT(config=sft_config_eval, use_wandb=False)

    # Save base model checkpoint (before any training)
    base_checkpoint_dir = Path("checkpoints/pipeline_stages/00_base_model")
    base_checkpoint_dir.mkdir(parents=True, exist_ok=True)
    base_checkpoint_path = base_checkpoint_dir / "base_model.pt"
    base_model_eval.save_checkpoint(str(base_checkpoint_path))
    print(f"✓ Saved base model checkpoint: {base_checkpoint_path}")

    # Extract test answers
    test_answers_for_eval = []
    for completion in sft_test_data["completions"]:
        answer = extract_answer_from_model_output(completion)
        test_answers_for_eval.append(answer if answer else "")

    print("\n" + "="*60)
    print("BASELINE EVALUATION (Before any training)")
    print("="*60)

    # Evaluate base model
    # base_metrics = evaluate_on_gsm8k(
    #     base_model_eval,
    #     sft_test_data["prompts"],
    #     test_answers_for_eval,
    #     len(sft_test_data["prompts"]),
    #     model_name="Base Model (Before Training)"
    # )

    # # Show examples
    # demonstrate_model_responses(
    #     base_model_eval,
    #     sft_test_data["prompts"],
    #     test_answers_for_eval,
    #     5,
    #     title="BASE MODEL EXAMPLES (Before Training)"
    # )

    # Clean up to save memory
    del base_model_eval
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()

    print("\n✓ Base model evaluation complete")

## 5. Supervised Fine-Tuning (SFT)

### 5.1 SFT Configuration

In [6]:
# SFT Configuration - Optimized for CoT dataset with proper gradient accumulation
sft_config = {
    "model": {
        "model_name": MODEL_NAME,
        "max_length": 512,         # shorter sequences for stability/perf
        "device": "mps",
        "model_type": "fp32",
    },
    "training": {
        "batch_size": 4,
        "learning_rate": 1e-5,     # conservative on MPS/FP32
        "num_epochs": 3,
        "gradient_accumulation_steps": 4,    # Effective batch = 4
        "max_grad_norm": 1.0,
        "warmup_steps": 20,        # ~10% warmup
        "mask_prompt": True,       # loss only on completion tokens
        "gradient_checkpointing": False,
        "weight_decay": 0.01,
        "label_smoothing": 0.00,
        "mixed_precision": {
            "enabled": False,
            "dtype": "fp32"
        }
    },
    "logging": {"log_interval": 10, "save_interval": 50},
    "validation": {"enabled": True, "interval": 20, "num_samples": 100},
    "wandb": {"enabled": False}
}

# Print configuration with calculated values for 500 CoT examples
effective_batch_size = sft_config['training']['batch_size'] * sft_config['training']['gradient_accumulation_steps']
steps_per_epoch = 500 // sft_config['training']['batch_size']  # 500 CoT examples
updates_per_epoch = steps_per_epoch // sft_config['training']['gradient_accumulation_steps']

print("SFT Configuration (for 500 CoT examples):")
print(json.dumps(sft_config, indent=2))
print(f"\n{'='*60}")
print("Calculated Training Parameters:")
print(f"{'='*60}")
print(f"Training dataset: 500 high-quality CoT examples")
print(f"Physical batch size: {sft_config['training']['batch_size']}")
print(f"Gradient accumulation steps: {sft_config['training']['gradient_accumulation_steps']}")
print(f"Effective batch size: {effective_batch_size}")
print(f"Steps per epoch: {steps_per_epoch}")
print(f"Optimizer updates per epoch: {updates_per_epoch}")
print(f"Total training steps: {steps_per_epoch * sft_config['training']['num_epochs']}")
print(f"{'='*60}")

SFT Configuration (for 500 CoT examples):
{
  "model": {
    "model_name": "Qwen/Qwen2.5-0.5B-Instruct",
    "max_length": 512,
    "device": "mps",
    "model_type": "fp32"
  },
  "training": {
    "batch_size": 4,
    "learning_rate": 1e-05,
    "num_epochs": 3,
    "gradient_accumulation_steps": 4,
    "max_grad_norm": 1.0,
    "warmup_steps": 20,
    "mask_prompt": true,
    "gradient_checkpointing": false,
    "weight_decay": 0.01,
    "label_smoothing": 0.0,
    "mixed_precision": {
      "enabled": false,
      "dtype": "fp32"
    }
  },
  "logging": {
    "log_interval": 10,
    "save_interval": 50
  },
  "validation": {
    "enabled": true,
    "interval": 20,
    "num_samples": 100
  },
  "wandb": {
    "enabled": false
  }
}

Calculated Training Parameters:
Training dataset: 500 high-quality CoT examples
Physical batch size: 4
Gradient accumulation steps: 4
Effective batch size: 16
Steps per epoch: 125
Optimizer updates per epoch: 31
Total training steps: 375


### 5.2 SFT Training

In [7]:
if RUN_SFT:
# Initialize SFT
    print("Initializing SFT...")
    sft = SFT(config=sft_config, use_wandb=False)

    print("\n" + "="*60)
    print("STARTING SFT TRAINING WITH HIGH-QUALITY CoT DATASET")
    print("="*60)
    print(f"Using {len(sft_train_data_cot['prompts'])} high-quality CoT examples (reference notebook dataset)")
    print(f"Note: Checkpoints will be auto-saved to checkpoints/sft/ every {sft_config['logging']['save_interval']} steps")
    print("="*60 + "\n")

    # Train the model with HIGH-QUALITY CoT dataset
    sft_train_data_cot["debug_boundary"] = True
    sft_results = sft.train(
        train_data=sft_train_data_cot,  # ← Using CoT dataset (500 examples)
        val_data=sft_val_data,
        num_episodes=sft_config['training']['num_epochs']
    )

    print("\n" + "="*60)
    print("SFT TRAINING COMPLETE")
    print("="*60)
    print(f"Total time: {sft_results['total_time']:.2f} seconds ({sft_results['total_time']/60:.2f} minutes)")
    print(f"Final loss: {sft_results['final_loss']:.4f}")

    # Save SFT checkpoint to pipeline stages folder
    print("\nSaving final checkpoint to pipeline_stages folder...")
    try:
        from pathlib import Path
        sft_checkpoint_dir = Path("checkpoints/pipeline_stages/01_after_sft")
        sft_checkpoint_dir.mkdir(parents=True, exist_ok=True)
        sft_checkpoint_path = sft_checkpoint_dir / "sft_complete.pt"
        
        sft.save_checkpoint(str(sft_checkpoint_path))
        
        # Verify the checkpoint was saved
        import os
        if os.path.exists(sft_checkpoint_path):
            file_size = os.path.getsize(sft_checkpoint_path) / (1024**3)  # GB
            print(f"✓ Saved SFT checkpoint: {sft_checkpoint_path}")
            print(f"  Checkpoint size: {file_size:.2f} GB")
        else:
            print(f"❌ ERROR: Checkpoint file was not created at {sft_checkpoint_path}")
    except Exception as e:
        print(f"❌ ERROR saving checkpoint: {e}")
        import traceback
        traceback.print_exc()
        raise

### 5.3 SFT Evaluation

In [8]:
if RUN_SFT:
    # Extract test answers for evaluation
    test_answers_for_eval = []
    for completion in sft_test_data["completions"]:
        answer = extract_answer_from_model_output(completion)
        test_answers_for_eval.append(answer if answer else "")

    print("\n" + "="*60)
    print("SFT MODEL EVALUATION")
    print("="*60)

    # Evaluate SFT model
    sft_metrics = evaluate_on_gsm8k(
        sft,
        sft_test_data["prompts"],
        test_answers_for_eval,
        len(sft_test_data["prompts"]),
        model_name="SFT Model (After Supervised Training)",
        save_results=True,
        results_file="results/sft_eval_results.json",
        step=0
    )

    # Show examples
    demonstrate_model_responses(
        sft,
        sft_test_data["prompts"],
        test_answers_for_eval,
        5,
        title="SFT MODEL EXAMPLES (After Supervised Training)"
    )

### 5.4 SFT Visualization

In [9]:
if RUN_SFT:
  print("Checking if model was trained...")

    # Generate a quick test
  test_prompt = """Respond in the following format:

    <reasoning>
    ...
    </reasoning>
    <answer>
    ...
    </answer>
    What is 2 + 2?"""

  response = sft.generate([test_prompt], max_new_tokens=200, temperature=0.7)[0]
  print(f"Model response:\n{response}")
  print(f"\nHas <answer> tag: {'<answer>' in response}")
  print(f"Has </answer> tag: {'</answer>' in response}")
  # Verify training data format
  print("Training data sample:")
  print(f"Prompt: {sft_train_data_cot['prompts'][0][:200]}")
  print(f"\nCompletion: {sft_train_data_cot['completions'][0][:200]}")

In [10]:
if RUN_SFT:
    # Extract metrics from SFT results
    sft_training_metrics = sft_results['training_metrics']
    sft_validation_metrics = sft_results['validation_metrics']

    # Plot SFT training metrics
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('SFT Training Metrics', fontsize=16)

    # Training Loss
    axes[0, 0].plot(sft_training_metrics["step"], sft_training_metrics["loss"], 'b-', alpha=0.7)
    axes[0, 0].set_xlabel('Training Step')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Training Loss')
    axes[0, 0].grid(True, alpha=0.3)

    # Learning Rate
    axes[0, 1].plot(sft_training_metrics["step"], sft_training_metrics["learning_rate"], 'g-', alpha=0.7)
    axes[0, 1].set_xlabel('Training Step')
    axes[0, 1].set_ylabel('Learning Rate')
    axes[0, 1].set_title('Learning Rate Schedule')
    axes[0, 1].grid(True, alpha=0.3)

    # Validation Loss
    if sft_validation_metrics["step"]:
        axes[1, 0].plot(sft_validation_metrics["step"], sft_validation_metrics["loss"], 
                        'ro-', markersize=8)
        axes[1, 0].set_xlabel('Training Step')
        axes[1, 0].set_ylabel('Loss')
        axes[1, 0].set_title('Validation Loss')
        axes[1, 0].grid(True, alpha=0.3)
    else:
        axes[1, 0].text(0.5, 0.5, 'No validation data', ha='center', va='center')

    # Tokens per Second
    axes[1, 1].plot(sft_training_metrics["step"], sft_training_metrics["tokens_per_second"], 
                    'orange', alpha=0.7, linewidth=2)
    axes[1, 1].set_xlabel('Training Step')
    axes[1, 1].set_ylabel('Tokens/Second')
    axes[1, 1].set_title('Processing Speed')
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Print SFT summary
    print("\n" + "="*60)
    print("SFT TRAINING SUMMARY")
    print("="*60)
    print(f"Final Loss: {sft_training_metrics['loss'][-1]:.4f}")
    print(f"Initial Loss: {sft_training_metrics['loss'][0]:.4f}")
    print(f"Loss Reduction: {sft_training_metrics['loss'][0] - sft_training_metrics['loss'][-1]:.4f}")
    print(f"Average Processing Speed: {np.mean(sft_training_metrics['tokens_per_second']):.1f} tokens/second")

In [11]:
if RUN_SFT:
    # MANUAL CHECKPOINT SAVE - Run this if SFT training completed but checkpoint wasn't saved
    try:
        # Check if sft variable exists
        print(f"SFT model in memory: {type(sft)}")
        print(f"SFT global step: {sft.global_step}")
        
        # Save the checkpoint manually
        from pathlib import Path
        sft_checkpoint_dir = Path("checkpoints/pipeline_stages/01_after_sft")
        sft_checkpoint_dir.mkdir(parents=True, exist_ok=True)
        sft_checkpoint_path = sft_checkpoint_dir / "sft_complete.pt"
        
        sft.save_checkpoint(str(sft_checkpoint_path))
        print(f"\n✓ Successfully saved SFT checkpoint to: {sft_checkpoint_path}")
        
        # Verify it exists
        import os
        if os.path.exists(sft_checkpoint_path):
            file_size = os.path.getsize(sft_checkpoint_path) / (1024**3)  # GB
            print(f"✓ Checkpoint file exists: {file_size:.2f} GB")
        else:
            print("❌ ERROR: Checkpoint file was not created!")
                
    except NameError:
        print("❌ ERROR: 'sft' variable not found in memory!")
        print("You need to re-run cell 13 (SFT Training) first.")
    except Exception as e:
        print(f"❌ ERROR saving checkpoint: {e}")
        import traceback
        traceback.print_exc()

## 7. Group Relative Policy Optimization (GRPO)

### 7.1 GRPO Configuration

In [12]:
# GRPO Configuration - Updated with working settings from overfit test
grpo_config = {
    "algorithm": {
        "name": "grpo",
        "group_size": 8,  # Keep at 8 (working well in overfit test)
        "kl_coef": 0.04,  # Increased from 0.1 → 0.15 for more stability
        "clip_epsilon": 0.2,
        "normalize_rewards": True,
        "store_completions": False,  # False for production (saves memory)
    },
    "training": {
        "batch_size": 16,  # Keep at 16 for full training (not 4 like overfit)
        "rollout_batch_size": 4,  # Keep at 4 for memory management
        "gradient_clip": 3.0,  # TIGHTENED from 1.0 → 0.1 for maximum stability
        "max_new_tokens": 500,
        "min_new_tokens": 50,  # LOWERED from 150 → 50 to allow </answer> early stopping
        "temperature": 0.6,#0.6,
        "num_episodes": 60,  # Full training episodes
        "minibatch_size": 32,  # Keep at 32 for full training
        "update_epochs": 1,
        "top_p": 0.9,#0.9,
        "entropy_coef": 0.005,  # Increased from 0.002 → 0.005 for more exploration
        "policy_loss_type": "token",
        "resample_batch_per_episode": True,  # ← CRITICAL: Set to True to disable fixed batch!
        # Clipping parameters (all validated in overfit test)
        "kl_estimator": "k3",
        "kl_clamp_min": -2.0,
        "kl_clamp_max": 2.0,
        "kl_reduction": "mean",
        "policy_log_ratio_clamp_min": -2.0,
        "policy_log_ratio_clamp_max": 2.0,
        "advantage_clip_min": -3.0,
        "advantage_clip_max": 3.0,
        "stop_sequences": ["</answer>"],
    },
    "model": {
        "max_length": 1024,
        "model_name": MODEL_NAME,
        "model_type": "fp32",
        "device": str(device),
        "compile": {
            "enabled": False,
            "backend": "aot_eager"
        }
    },
    "device_optimizations": {
        "clear_cache_on_mps": True,  # ENABLED for M4 unified memory
    },
    "logging": {
        "log_interval": 1,
        "save_interval": 15,
        "show_trajectory_progress": True,
    },
    "validation": {
        "enabled": True,
        "interval": 20,
        "num_samples": 50,
        "num_demo_examples": 5,
    },
    "wandb": {
        "enabled": False,
    },
    "optimizer": {
        "type": "adamw",
        "lr": 5e-6,  # DECREASED from 3e-6 → 1e-6 for more stability
        "weight_decay": 0.01,
        "betas": (0.9, 0.999),
        "eps": 1e-8,
        "fused": False,
        # INCREASED warmup for stability
        "warmup_steps": 0,  # Increased from 10 → 30
        "warmup_start_lr": 1e-9,  # Decreased from 1e-8 → 1e-9
        "warmup_type": "linear"
    },
    "optimization": {
        "mixed_precision": {
            "enabled": False,
            "dtype": "fp32",
            "mode": "fp32"
        }
    },
    "timing": {
        "enabled": False,
    },
    "debug": {
        "enabled": False,  # Disabled for production training
        "log_dir": "debug_logs",
        "loss": False,
        "advantages": False,
        "gradients": False,
        "generation": False,
        "alignment": False,
    }
}

print("="*70)
print("GRPO Configuration - UPDATED WITH OVERFIT TEST SETTINGS")
print("="*70)
print("\n📊 KEY CHANGES FROM OVERFIT TEST:")
print("  ✅ KL coefficient:     0.10 → 0.15 (more stability)")
print("  ✅ Gradient clip:      1.0 → 0.1 (MUCH tighter for stability)")
print("  ✅ Min new tokens:     150 → 50 (allow </answer> early stopping)")
print("  ✅ Entropy coef:       0.002 → 0.005 (more exploration)")
print("  ✅ Learning rate:      3e-6 → 1e-6 (more conservative)")
print("  ✅ Warmup steps:       10 → 30 (longer warmup)")
print("  ✅ Warmup start LR:    1e-8 → 1e-9 (lower start)")
print("  ✅ RESAMPLE BATCH:     ENABLED (normal training mode - not fixed batch!)")
print("\n📊 KEPT FOR FULL TRAINING (not from overfit):")
print("  • Batch size:          16 (not 4 - need more data coverage)")
print("  • Minibatch size:      32 (not 8 - better for full training)")
print("  • Num episodes:        500 (not 100 - full training run)")
print("\n⚠️  STOPPING BEHAVIOR:")
print("  • min_new_tokens=50: Forces at least 50 tokens before stopping")
print("  • stop_sequences=['</answer>']: Stops when </answer> appears")
print("  • Result: Model generates 50-400 tokens, stops at </answer> if present")
print("\n🧹 MEMORY MANAGEMENT:")
print("  • Cache clearing:      ENABLED on MPS (M4 unified memory)")
print("  • Garbage collection:  Forced every episode")
print("  • Timing data:         Auto-reset every 10 episodes")
print("="*70)

print("\nGRPO Configuration:")
print(json.dumps(grpo_config, indent=2))

GRPO Configuration - UPDATED WITH OVERFIT TEST SETTINGS

📊 KEY CHANGES FROM OVERFIT TEST:
  ✅ KL coefficient:     0.10 → 0.15 (more stability)
  ✅ Gradient clip:      1.0 → 0.1 (MUCH tighter for stability)
  ✅ Min new tokens:     150 → 50 (allow </answer> early stopping)
  ✅ Entropy coef:       0.002 → 0.005 (more exploration)
  ✅ Learning rate:      3e-6 → 1e-6 (more conservative)
  ✅ Warmup steps:       10 → 30 (longer warmup)
  ✅ Warmup start LR:    1e-8 → 1e-9 (lower start)
  ✅ RESAMPLE BATCH:     ENABLED (normal training mode - not fixed batch!)

📊 KEPT FOR FULL TRAINING (not from overfit):
  • Batch size:          16 (not 4 - need more data coverage)
  • Minibatch size:      32 (not 8 - better for full training)
  • Num episodes:        500 (not 100 - full training run)

⚠️  STOPPING BEHAVIOR:
  • min_new_tokens=50: Forces at least 50 tokens before stopping
  • stop_sequences=['</answer>']: Stops when </answer> appears
  • Result: Model generates 50-400 tokens, stops at </answer>

### 7.2 Initialize GRPO with SFT Checkpoint

In [13]:
# Initialize GRPO with Checkpoint (SFT or Resume)
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig

if CONTINUE_FROM > 0:
    # Resume from GRPO checkpoint
    print("\n" + "="*60)
    print(f"RESUMING FROM GRPO CHECKPOINT (Episode {CONTINUE_FROM})")
    print("="*60)
    
    grpo_checkpoint_path = Path(f"checkpoints/grpo_qwen_math/checkpoint_episode_{CONTINUE_FROM}.pt")
    print(f"Checkpoint path: {grpo_checkpoint_path}")
    
    # Load GRPO checkpoint
    grpo_checkpoint = torch.load(grpo_checkpoint_path, map_location=device)
    
    # DEBUG: Show comprehensive checkpoint contents
    print("\n" + "="*60)
    print("CHECKPOINT CONTENTS:")
    print("="*60)
    print(f"Keys in checkpoint: {list(grpo_checkpoint.keys())}")
    print(f"Episode: {grpo_checkpoint.get('episode')}")
    print(f"Total steps: {grpo_checkpoint.get('total_steps')}")
    print(f"Current episode: {grpo_checkpoint.get('current_episode')}")
    
    # Show optimizer config from checkpoint
    opt_state = grpo_checkpoint["optimizer_state_dict"]
    print(f"\nOptimizer state keys: {list(opt_state.keys())}")
    if "param_groups" in opt_state:
        pg = opt_state["param_groups"][0]
        print(f"Optimizer param_group[0]:")
        for key in ['lr', 'betas', 'eps', 'weight_decay']:
            if key in pg:
                print(f"  {key}: {pg[key]}")
    
    # Show scheduler state from checkpoint
    sched_state = grpo_checkpoint.get("scheduler_state_dict")
    print(f"\nScheduler state: {sched_state}")
    
    # Show optimizer config from checkpoint's config
    checkpoint_config = grpo_checkpoint.get("config", {})
    checkpoint_opt_config = checkpoint_config.get("optimizer", {})
    print(f"\nOptimizer config (from checkpoint):")
    print(f"  lr: {checkpoint_opt_config.get('lr')}")
    print(f"  warmup_steps: {checkpoint_opt_config.get('warmup_steps')}")
    print(f"  warmup_start_lr: {checkpoint_opt_config.get('warmup_start_lr')}")
    print("="*60)
    
    model_name = checkpoint_config["model"]["model_name"]
    print(f"\nModel: {model_name}")
    print(f"Resuming from episode: {grpo_checkpoint['episode']}")
    
    # Load tokenizer only (lightweight, no weights)
    print("\nLoading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print("✓ Tokenizer loaded")
    
    # Load model architecture without pretrained weights (much faster!)
    print("\nLoading model architecture from config...")
    config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModelForCausalLM.from_config(config, trust_remote_code=True)
    model = model.to(dtype=torch.float32)
    print("✓ Model architecture loaded (no pretrained weights downloaded)")
    
    # Load trained weights from GRPO checkpoint
    # GRPO wraps model in LanguageModel class, so keys have "model." prefix that needs to be removed
    print("\nLoading trained weights from checkpoint...")
    policy_state = grpo_checkpoint["policy_state_dict"]
    
    # Strip "model." prefix from keys (e.g., "model.model.embed_tokens.weight" -> "model.embed_tokens.weight")
    model_state = {k.replace("model.", "", 1): v for k, v in policy_state.items() if k.startswith("model.")}
    
    model.load_state_dict(model_state)
    print("✓ Checkpoint weights loaded successfully")
    
    # Initialize GRPO
    print("\nInitializing GRPO and restoring optimizer/scheduler state...")
    grpo = GRPO(
        config=grpo_config,
        batch_reward_fn=compute_math_rewards_batch,
        model=model,
        tokenizer=tokenizer,
        use_wandb=False
    )
    
    # Restore optimizer state (momentum, variance, AND learning rate from checkpoint)
    grpo.optimizer.load_state_dict(grpo_checkpoint["optimizer_state_dict"])
    checkpoint_lr = grpo.optimizer.param_groups[0]['lr']

    print(f"  ✓ Optimizer state restored")
    print(f"  ✓ Learning rate from checkpoint: {checkpoint_lr:.2e}")

    # CRITICAL: Disable the scheduler when resuming!
    # At episode 45, we're WAY past warmup (30 steps). We should continue with the checkpoint's LR.
    # The scheduler was created fresh in GRPO.__init__ and would restart warmup - we don't want that!
    total_steps = grpo_checkpoint.get("total_steps", 0)
    warmup_steps = grpo_config["optimizer"]["warmup_steps"]

    if total_steps >= warmup_steps:
        # Past warmup - disable scheduler entirely
        grpo.lr_scheduler = None
        print(f"  ✓ Scheduler disabled (past warmup: step {total_steps}/{warmup_steps})")
        print(f"  ✓ Will continue with fixed LR = {checkpoint_lr:.2e}")
    else:
        # Still in warmup phase - need to advance scheduler to correct position
        print(f"  ⚠️  Still in warmup phase (step {total_steps}/{warmup_steps})")
        print(f"  ⚠️  Advancing scheduler to step {total_steps}...")

        # Step the scheduler to catch up to where we were
        for _ in range(total_steps):
            grpo.lr_scheduler.step()

        final_lr = grpo.optimizer.param_groups[0]['lr']
        print(f"  ✓ Scheduler advanced, current LR: {final_lr:.2e}")

    # CRITICAL: Restore reference policy (the original SFT model, not a copy of trained policy!)
    if "ref_policy_state_dict" in grpo_checkpoint and grpo_checkpoint["ref_policy_state_dict"] is not None:
        grpo.ref_policy.load_state_dict(grpo_checkpoint["ref_policy_state_dict"])
        print(f"  ✓ Reference policy restored (KL constraint preserved)")
    else:
        print(f"  ⚠️  WARNING: No reference policy in checkpoint - KL divergence will be incorrect!")
    
    # Restore total_steps for proper tracking
    if "total_steps" in grpo_checkpoint:
        grpo.total_steps = grpo_checkpoint["total_steps"]
        print(f"  ✓ Training steps restored: {grpo.total_steps}")
    
    # Restore episode counter
    # Checkpoint contains the LAST completed episode, so resume at NEXT episode
    # E.g., checkpoint_episode_45.pt has episode=44, so we start at 45
    last_completed_episode = grpo_checkpoint["episode"]
    grpo.current_episode = last_completed_episode + 1
    
    print(f"  Last completed episode in checkpoint: {last_completed_episode}")
    print(f"  Will resume training from episode: {grpo.current_episode}")
    
    print("\n" + "="*60)
    print(f"✓ GRPO successfully resumed from episode {CONTINUE_FROM}")
    print("="*60)
else:
    # Start from SFT checkpoint or base model
    print("\n" + "="*60)
    print("Loading SFT checkpoint and initializing GRPO")
    print("="*60)
    print(f"Checkpoint path: {sft_checkpoint_path}")
    
    # Load SFT checkpoint
    sft_checkpoint = torch.load(sft_checkpoint_path, map_location=device)
    model_name = sft_checkpoint["config"]["model"]["model_name"]
    print(f"Model: {model_name}")
    
    # Load fresh model and tokenizer
    print("\nLoading fresh model and tokenizer...")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.float32,
        trust_remote_code=True
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    print("✓ Model and tokenizer loaded")
    
    # Load SFT weights into fresh model
    print("\nLoading SFT weights into model...")
    model.load_state_dict(sft_checkpoint["model_state_dict"])
    print("✓ SFT weights loaded successfully")
    
    # Initialize GRPO with pre-loaded model
    print("\nInitializing GRPO with SFT-trained model...")
    grpo = GRPO(
        config=grpo_config,
        batch_reward_fn=compute_math_rewards_batch,
        model=model,
        tokenizer=tokenizer,
        use_wandb=False
    )
    
    print("\n" + "="*60)
    print("✓ GRPO successfully initialized with SFT-trained model")
    print("="*60)
    print("The reference model has also been initialized from the SFT weights.")
    print("Ready to start GRPO training!")


Loading SFT checkpoint and initializing GRPO
Checkpoint path: checkpoints/pipeline_stages/01_after_sft/sft_complete.pt
Model: Qwen/Qwen2.5-0.5B-Instruct

Loading fresh model and tokenizer...
✓ Model and tokenizer loaded

Loading SFT weights into model...
✓ SFT weights loaded successfully

Initializing GRPO with SFT-trained model...


/Users/tomassikora/WORK/programing/VUT/simple_rl/.venv/lib/python3.11/site-packages/torch/__init__.py:1615: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/Context.cpp:50.)
  _C._set_float32_matmul_precision(precision)


✓ GRPO initialized with model: pre-loaded model
  Total parameters: 494,032,768
  Trainable parameters: 494,032,768
  Device: mps
✓ Reference model frozen: 290 parameters
✓ KL divergence estimator: k3
  KL clamp range: [-2.0, 2.0]
✓ Policy loss type: token
  Token-level normalization (10-100x lower gradients on long sequences)
✓ Clipping parameters:
  PPO ratio clipping: [0.8, 1.2]
  Advantages: [-3.0, 3.0]

✓ GRPO successfully initialized with SFT-trained model
The reference model has also been initialized from the SFT weights.
Ready to start GRPO training!


### 7.3 GRPO Training

In [ ]:
# Prepare data for GRPO
grpo_train_data = {
    "prompts": grpo_train_prompts,
    "answers": grpo_train_answers
}

grpo_val_data = {
    "prompts": grpo_val_prompts,
    "answers": grpo_val_answers
}

print("\n" + "="*60)
print("STARTING GRPO TRAINING")
print("="*60)

# Train GRPO
grpo_results = grpo.train(
    train_data=grpo_train_data,
    val_data=grpo_val_data,
    num_episodes=grpo_config['training']['num_episodes']
)

print("\n" + "="*60)
print("GRPO TRAINING COMPLETE")
print("="*60)
print(f"Total time: {grpo_results['total_time']:.2f} seconds ({grpo_results['total_time']/60:.2f} minutes)")
print(f"Total tokens processed: {grpo_results['training_metrics']['total_tokens'][-1]:,}")
print(f"Final reward: {grpo_results['final_reward']:.3f}")

# Organize GRPO checkpoints into pipeline stages folder
import shutil

print("\n" + "="*60)
print("ORGANIZING GRPO CHECKPOINTS")
print("="*60)

# Create pipeline stages directories
grpo_checkpoints = [
    (15, "02_grpo_25_percent"),
    (30, "03_grpo_50_percent"),
    (45, "04_grpo_75_percent"),
    (60, "05_grpo_100_percent")
]

source_checkpoint_dir = Path("checkpoints/grpo_qwen_math")
pipeline_stages_dir = Path("checkpoints/pipeline_stages")

for episode, stage_name in grpo_checkpoints:
    source_path = source_checkpoint_dir / f"checkpoint_episode_{episode}.pt"
    dest_dir = pipeline_stages_dir / stage_name
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_path = dest_dir / f"grpo_{episode}_episodes.pt"
    
    if source_path.exists():
        shutil.copy2(source_path, dest_path)
        print(f"✓ Copied episode {episode} → {dest_path}")
    else:
        print(f"⚠ Warning: {source_path} not found")

print("\n" + "="*60)
print("CHECKPOINT ORGANIZATION COMPLETE")
print("="*60)
print(f"\nAll checkpoints saved in: checkpoints/pipeline_stages/")
print(f"  00_base_model/         - Base model (before training)")
print(f"  01_after_sft/          - After SFT completion")
print(f"  02_grpo_25_percent/    - GRPO at 25% (15 episodes)")
print(f"  03_grpo_50_percent/    - GRPO at 50% (30 episodes)")
print(f"  04_grpo_75_percent/    - GRPO at 75% (45 episodes)")
print(f"  05_grpo_100_percent/   - GRPO at 100% (60 episodes)")


STARTING GRPO TRAINING
Starting proper GRPO training for 60 episodes...
  - Using PPO clipped objective (clip_epsilon=0.2)
  - Training samples: 6973
  - Batch size: 16
  - Update epochs: 1
  - Group size: 8
  - Rollout batch size: 4 prompts (OOM prevention)
  - Minibatch size: 32 rollouts per update
  - Validation enabled: every 20 episodes
    - Evaluating 50 validation samples
    - Showing 5 demo examples
  - Resampling batch every episode (normal training mode)
  [Trajectory 1/4] ✓ Decoded completion matches stored completion

✓ [Episode 0] Text reconstruction verified for first sequence
  Prompt length: 346 chars
  Completion length: 1270 chars
  Full generation length: 1616 chars
  Completion IDs shape: torch.Size([32, 500])

[DEBUG] RIGHT AFTER compute_log_probs:
  generated_ids shape: torch.Size([32, 647])
  generated_mask shape: torch.Size([32, 647])
  policy_log_probs shape: torch.Size([32, 500])
  logits_to_keep (max_completion_len): 500
  completion_ids shape: torch.Size(

### 7.4 GRPO Evaluation

In [ ]:
print("\n" + "="*60)
print("GRPO MODEL EVALUATION")
print("="*60)

# Evaluate GRPO model
grpo_metrics = evaluate_on_gsm8k(
    grpo,
    grpo_test_prompts,
    grpo_test_answers,
    len(grpo_test_prompts),
    model_name="GRPO Model (After RL Training)",
    save_results=True,
    results_file="results/grpo_eval_results.json",
    step=grpo_config["training"]["num_episodes"]
)

# Show examples
demonstrate_model_responses(
    grpo,
    grpo_test_prompts,
    grpo_test_answers,
    5,
    title="GRPO MODEL EXAMPLES (After RL Training)"
)

### 7.5 GRPO Visualization

In [15]:
# Extract metrics from GRPO results
grpo_training_metrics = grpo_results['training_metrics']
grpo_validation_metrics = grpo_results['validation_metrics']

# Plot GRPO training and validation metrics
fig, axes = plt.subplots(4, 2, figsize=(14, 16))
fig.suptitle('GRPO Training and Validation Metrics', fontsize=16)

# Total Loss
axes[0, 0].plot(grpo_training_metrics["episode"], grpo_training_metrics["total_loss"], 'b-', alpha=0.7)
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel('Total Loss')
axes[0, 0].set_title('Total Loss over Training')
axes[0, 0].grid(True, alpha=0.3)

# Mean Reward
axes[0, 1].plot(grpo_training_metrics["episode"], grpo_training_metrics["reward_mean"], 'purple', alpha=0.7, label='Mean')
axes[0, 1].fill_between(
    grpo_training_metrics["episode"],
    np.array(grpo_training_metrics["reward_mean"]) - np.array(grpo_training_metrics["reward_std"]),
    np.array(grpo_training_metrics["reward_mean"]) + np.array(grpo_training_metrics["reward_std"]),
    alpha=0.3, color='purple'
)
axes[0, 1].set_xlabel('Episode')
axes[0, 1].set_ylabel('Reward')
axes[0, 1].set_title('Mean Reward ± Std')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

# KL Divergence
axes[1, 0].plot(grpo_training_metrics["episode"], grpo_training_metrics["kl_divergence"], 'r-', alpha=0.7)
axes[1, 0].set_xlabel('Episode')
axes[1, 0].set_ylabel('KL Divergence')
axes[1, 0].set_title('KL Divergence from Reference')
axes[1, 0].grid(True, alpha=0.3)

# Validation Accuracy
if grpo_validation_metrics["episode"]:
    axes[1, 1].plot(grpo_validation_metrics["episode"], grpo_validation_metrics["exact_accuracy"], 
                    'go-', label='Exact Match', markersize=8)
    axes[1, 1].plot(grpo_validation_metrics["episode"], grpo_validation_metrics["numeric_accuracy"], 
                    'bs-', label='Numeric Match', markersize=8)
    axes[1, 1].set_xlabel('Episode')
    axes[1, 1].set_ylabel('Accuracy (%)')
    axes[1, 1].set_title('Validation Accuracy')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
else:
    axes[1, 1].text(0.5, 0.5, 'No validation data', ha='center', va='center')

# Format Compliance
if grpo_validation_metrics["episode"]:
    axes[2, 0].plot(grpo_validation_metrics["episode"], grpo_validation_metrics["format_compliance"], 
                    'mo-', label='Format Compliance %', markersize=8)
    axes[2, 0].set_xlabel('Episode')
    axes[2, 0].set_ylabel('Compliance (%)')
    axes[2, 0].set_title('Format Compliance')
    axes[2, 0].legend()
    axes[2, 0].grid(True, alpha=0.3)
else:
    axes[2, 0].text(0.5, 0.5, 'No validation data', ha='center', va='center')

# Episode Time
axes[2, 1].plot(grpo_training_metrics["episode"], grpo_training_metrics["episode_time"], 
                'orange', alpha=0.7, linewidth=2)
axes[2, 1].set_xlabel('Episode')
axes[2, 1].set_ylabel('Time (seconds)')
axes[2, 1].set_title('Episode Training Time')
axes[2, 1].grid(True, alpha=0.3)

# Cumulative Tokens Processed
axes[3, 0].plot(grpo_training_metrics["episode"], 
                [t/1000 for t in grpo_training_metrics["total_tokens"]], 
                'green', alpha=0.7, linewidth=2)
axes[3, 0].set_xlabel('Episode')
axes[3, 0].set_ylabel('Tokens (thousands)')
axes[3, 0].set_title('Cumulative Tokens Processed')
axes[3, 0].grid(True, alpha=0.3)

# Processing Speed
axes[3, 1].plot(grpo_training_metrics["episode"], grpo_training_metrics["tokens_per_second"], 
                'darkblue', alpha=0.7, linewidth=2)
axes[3, 1].set_xlabel('Episode')
axes[3, 1].set_ylabel('Tokens/Second')
axes[3, 1].set_title('Token Processing Speed')
axes[3, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print GRPO summary
print("\n" + "="*60)
print("GRPO TRAINING SUMMARY")
print("="*60)
print(f"Final Total Loss: {grpo_training_metrics['total_loss'][-1]:.4f}")
print(f"Final KL Divergence: {grpo_training_metrics['kl_divergence'][-1]:.4f}")
print(f"Final Mean Reward: {grpo_training_metrics['reward_mean'][-1]:.3f}")
print(f"Average Reward (last 5 episodes): {np.mean(grpo_training_metrics['reward_mean'][-5:]):.3f}")
print(f"Total Tokens Processed: {grpo_training_metrics['total_tokens'][-1]:,}")
print(f"Average Processing Speed: {np.mean(grpo_training_metrics['tokens_per_second']):.1f} tokens/second")

NameError: name 'grpo_results' is not defined

## 8. Final Comparison: Base vs SFT vs GRPO

In [ ]:
# Compare all three models
print("\n" + "="*80)
print("FINAL COMPARISON: BASE MODEL vs SFT vs GRPO")
print("="*80)

comparison_data = {
    'Model': ['Base (Before Training)', 'SFT (After Supervised)', 'GRPO (After RL)'],
    'Exact Accuracy': [
        f"{base_metrics['exact_accuracy']:.1f}%",
        f"{sft_metrics['exact_accuracy']:.1f}%",
        f"{grpo_metrics['exact_accuracy']:.1f}%"
    ],
    'Numeric Accuracy': [
        f"{base_metrics['numeric_accuracy']:.1f}%",
        f"{sft_metrics['numeric_accuracy']:.1f}%",
        f"{grpo_metrics['numeric_accuracy']:.1f}%"
    ],
    'Format Compliance': [
        f"{base_metrics['format_compliance']:.1f}%",
        f"{sft_metrics['format_compliance']:.1f}%",
        f"{grpo_metrics['format_compliance']:.1f}%"
    ],
    'Avg Format Score': [
        f"{base_metrics['avg_format_score']:.3f}",
        f"{sft_metrics['avg_format_score']:.3f}",
        f"{grpo_metrics['avg_format_score']:.3f}"
    ],
    'Avg Correctness': [
        f"{base_metrics['avg_correctness_score']:.3f}",
        f"{sft_metrics['avg_correctness_score']:.3f}",
        f"{grpo_metrics['avg_correctness_score']:.3f}"
    ]
}

# Print table
print(f"\n{'Model':<30} {'Exact Acc':<12} {'Numeric Acc':<12} {'Format':<12} {'Fmt Score':<12} {'Correct':<12}")
print("-" * 90)
for i in range(3):
    print(f"{comparison_data['Model'][i]:<30} "
          f"{comparison_data['Exact Accuracy'][i]:<12} "
          f"{comparison_data['Numeric Accuracy'][i]:<12} "
          f"{comparison_data['Format Compliance'][i]:<12} "
          f"{comparison_data['Avg Format Score'][i]:<12} "
          f"{comparison_data['Avg Correctness'][i]:<12}")

# Calculate improvements
print("\n" + "="*80)
print("IMPROVEMENTS")
print("="*80)
print(f"\nBase → SFT:")
print(f"  Exact Accuracy: {sft_metrics['exact_accuracy'] - base_metrics['exact_accuracy']:+.1f}%")
print(f"  Format Compliance: {sft_metrics['format_compliance'] - base_metrics['format_compliance']:+.1f}%")
print(f"  Correctness Score: {sft_metrics['avg_correctness_score'] - base_metrics['avg_correctness_score']:+.3f}")

print(f"\nSFT → GRPO:")
print(f"  Exact Accuracy: {grpo_metrics['exact_accuracy'] - sft_metrics['exact_accuracy']:+.1f}%")
print(f"  Format Compliance: {grpo_metrics['format_compliance'] - sft_metrics['format_compliance']:+.1f}%")
print(f"  Correctness Score: {grpo_metrics['avg_correctness_score'] - sft_metrics['avg_correctness_score']:+.3f}")

print(f"\nBase → GRPO (Total):")
print(f"  Exact Accuracy: {grpo_metrics['exact_accuracy'] - base_metrics['exact_accuracy']:+.1f}%")
print(f"  Format Compliance: {grpo_metrics['format_compliance'] - base_metrics['format_compliance']:+.1f}%")
print(f"  Correctness Score: {grpo_metrics['avg_correctness_score'] - base_metrics['avg_correctness_score']:+.3f}")

print("\n" + "="*80)
print("FULL PIPELINE COMPLETE")
print("="*80)
print(f"\nTraining Configuration:")
print(f"  SFT Training: 200 examples (train[:200])")
print(f"  GRPO Training: 200 examples (train[200:400]) - NO OVERLAP")
print(f"  Validation: 1000 examples (same for both)")
print(f"  Test: 200 examples (same for both)")
print(f"\nTraining Time:")
print(f"  SFT Training: {sft_results['total_time']/60:.2f} minutes")
print(f"  GRPO Training: {grpo_results['total_time']/60:.2f} minutes")
print(f"  Total Training: {(sft_results['total_time'] + grpo_results['total_time'])/60:.2f} minutes")

In [ ]:
# ============================================================
# Close Logging
# ============================================================
close_logger(logger)

logger.info("="*60)
logger.info(f"✓ Training log saved to: {log_path}")
logger.info("="*60)